# GRACE Parameter Explorer

An interactive tool for choosing the GRACE parameters $(\beta, \tau)$ that
match *your* domain's accuracy/speed preferences.

GRACE unifies task **accuracy** $A$ and **time efficiency** into one score:

$$\mathrm{GRACE}(A, r;\,\beta,\tau) = \frac{(1+\beta^2)\,A\,T^*(r)}{\beta^2 A + T^*(r)},
\qquad T^*(r) = \frac{1}{1+r^2}\,e^{-\tau\max(0,\,r-1)}$$

where $r = t_\text{total}/t_\text{max}$ is the normalized runtime ($r=1$ means
the agent used exactly its time budget).

* **$\beta$** sets the accuracy/speed balance (F-beta convention): $\beta=1$ weights
  them equally, $\beta<1$ emphasizes **accuracy**, $\beta>1$ emphasizes **speed**.
* **$\tau$** sets deadline strictness: $\tau=0$ is permissive (no extra penalty past
  the deadline); larger $\tau$ punishes overruns $r>1$ harder.

**How to use this notebook**

1. Run all cells (`Kernel -> Restart & Run All`).
2. Drag the sliders in each section. The score landscape, time-penalty curve,
   and trade-off contours redraw live.
3. When the landscape looks right for your application, copy the parameters from
   the **Export** cell at the bottom.

**This notebook is self-contained.** It runs on **Google Colab** with no setup:
just upload it (*File -> Upload notebook*) and *Runtime -> Run all*. Colab already
ships `numpy`, `matplotlib`, and `ipywidgets`. It will use the `grace_metric`
package if it happens to be installed, but otherwise falls back to a built-in copy
of the published formulas, so nothing needs to be installed or cloned.

In [ ]:
# --- Dependencies (no-op on Colab; installs only if something is missing) ---
import importlib, subprocess, sys
for _pkg in ("numpy", "matplotlib", "ipywidgets"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
print("dependencies ready")

In [ ]:
# --- The GRACE metric --------------------------------------------------
# Use the installed grace_metric package if present (e.g. from a checkout or
# `pip install grace_metric`); otherwise fall back to this inline copy of the
# published formulas so the notebook is fully self-contained on Colab.
import math
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Match the paper's GRACE heatmaps: inferno colormap on a fixed [0, 1] scale.
NORM = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

try:
    from grace_metric import grace, t_star, weights_to_params, REGIMES
    _SOURCE = "grace_metric package"
except ModuleNotFoundError:
    # --- inline fallback: mirrors grace_metric/core.py and regimes.py ---------
    def t_star(r, tau=0.0):
        # T*(r) = 1/(1+r^2) * exp(-tau * max(0, r-1)).
        return (1.0 / (1.0 + r ** 2)) * math.exp(-tau * max(0.0, r - 1.0))

    def grace(A, r, beta=1.0, tau=0.0):
        # GRACE(A,r;beta,tau) = (1+beta^2)*A*T* / (beta^2*A + T*).
        Ts = t_star(r, tau)
        num = (1.0 + beta ** 2) * A * Ts
        den = beta ** 2 * A + Ts
        return 0.0 if den == 0 else num / den

    def weights_to_params(w_beta, w_tau):
        # Map preference weights in (0,1) to (beta, tau).
        return math.sqrt(w_beta / (1.0 - w_beta)), w_tau / (1.0 - w_tau)

    REGIMES = {
        "permissive": {"beta": 1.0, "tau": 0.0},
        "moderate":   {"beta": 1.0, "tau": 0.43},
        "strict":     {"beta": 2.0, "tau": 5.67},
    }
    _SOURCE = "inline self-contained copy"

# Vectorized versions for fast grid evaluation (identical values on arrays).
def t_star_vec(r, tau):
    return (1.0 / (1.0 + r ** 2)) * np.exp(-tau * np.maximum(0.0, r - 1.0))

def grace_vec(A, r, beta, tau):
    Ts = t_star_vec(r, tau)
    num = (1.0 + beta ** 2) * A * Ts
    den = beta ** 2 * A + Ts
    return np.divide(num, den, out=np.zeros_like(num, dtype=float), where=den != 0)

# Sanity check: vectorized == scalar reference implementation.
for _A, _r, _b, _t in [(0.8, 1.5, 1.0, 0.43), (0.0, 0.0, 1.0, 0.0),
                       (0.8, 0.0, 0.0, 0.0), (1.0, 2.0, 2.0, 5.67)]:
    assert abs(grace_vec(np.array([_A]), np.array([_r]), _b, _t)[0]
               - grace(_A, _r, _b, _t)) < 1e-12

print(f"GRACE metric ready (source: {_SOURCE}). Named regimes:")
for name, p in REGIMES.items():
    print(f"  {name:11s} beta={p['beta']:.2f}  tau={p['tau']:.2f}")

## 1. Score landscape

The filled contour shows the GRACE score over every combination of normalized
runtime $r$ (x-axis) and accuracy $A$ (y-axis) for the current $(\beta,\tau)$.
The white star marks your chosen operating point $(A_0, r_0)$ -- e.g. a method
that scores $A_0$ at runtime $r_0$.

* Slide **$\beta$** and watch the iso-score lines tilt between favoring accuracy
  and favoring speed.
* Slide **$\tau$** and watch the score collapse to the right of the deadline
  line $r=1$.
* The side panel reports the score at your operating point and how the three
  named regimes would rate the same point.

> Note: $A=0$ forces $\mathrm{GRACE}=0$ at any runtime (you cannot be efficient at
> a task you never complete), so the bottom edge of the landscape is always zero.

In [ ]:
# --- Interactive score landscape --------------------------------------
s_beta = widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.05,
            description="beta", continuous_update=True, readout_format=".2f")
s_tau  = widgets.FloatSlider(value=0.0, min=0.0, max=8.0, step=0.05,
            description="tau", continuous_update=True, readout_format=".2f")
s_A0   = widgets.FloatSlider(value=0.80, min=0.0, max=1.0, step=0.01,
            description="A0 (acc)", continuous_update=True, readout_format=".2f")
s_r0   = widgets.FloatSlider(value=1.20, min=0.0, max=3.0, step=0.05,
            description="r0 (runtime)", continuous_update=True, readout_format=".2f")
s_rmax = widgets.FloatSlider(value=2.5, min=1.0, max=4.0, step=0.5,
            description="r-axis max", continuous_update=False, readout_format=".1f")
c_regimes = widgets.Checkbox(value=True, description="compare named regimes")

controls = widgets.VBox([
    widgets.HTML("<b>Parameters</b>"), s_beta, s_tau,
    widgets.HTML("<b>Operating point</b>"), s_A0, s_r0,
    widgets.HTML("<b>View</b>"), s_rmax, c_regimes,
], layout=widgets.Layout(width="320px"))
out1 = widgets.Output()

def draw_landscape(*_):
    beta, tau = s_beta.value, s_tau.value
    A0, r0, rmax = s_A0.value, s_r0.value, s_rmax.value
    A = np.linspace(0, 1, 60)        # vertical axis
    r = np.linspace(0, rmax, 80)     # horizontal axis
    RR, AA = np.meshgrid(r, A)       # rows indexed by A, columns by r
    Z = grace_vec(AA, RR, beta, tau)
    with out1:
        out1.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.6, 4.6))
        levels = np.linspace(0, 1, 21)
        cf = ax.contourf(RR, AA, Z, levels=levels, cmap="inferno", norm=NORM)
        cs = ax.contour(RR, AA, Z, levels=[0.2, 0.4, 0.6, 0.8], colors="white",
                        linestyles="dashed", linewidths=0.6, alpha=0.5)
        ax.clabel(cs, inline=True, fontsize=7, fmt="%.1f")
        ax.axvline(1.0, color="white", ls=":", lw=1.0, alpha=0.7)
        ax.text(1.02, 0.03, "deadline r=1", color="white", fontsize=8)
        ax.scatter([r0], [A0], s=180, marker="*", color="white",
                   edgecolor="black", zorder=5)
        g0 = grace(A0, r0, beta, tau)
        ax.annotate(f"GRACE={g0:.3f}", (r0, A0), color="white", fontsize=9,
                    xytext=(6, 6), textcoords="offset points",
                    bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5))
        ax.set_xlabel("normalized runtime  r"); ax.set_ylabel("accuracy  A")
        ax.set_title(f"GRACE landscape  (beta={beta:.2f}, tau={tau:.2f})")
        fig.colorbar(cf, ax=ax, label="GRACE score")
        plt.tight_layout(); plt.show()

        wb = beta ** 2 / (1.0 + beta ** 2)  # inverse of weights_to_params
        print(f"At operating point  A0={A0:.2f}, r0={r0:.2f}:")
        print(f"  T*(r0)        = {t_star(r0, tau):.3f}")
        print(f"  GRACE         = {g0:.3f}")
        print(f"  (beta={beta:.2f} <=> speed-weight w_beta={wb:.2f};  "
              f"beta<1 favors accuracy, beta>1 favors speed)")
        if c_regimes.value:
            print("\n  Same point under the named regimes:")
            for name, p in REGIMES.items():
                print(f"    {name:11s} (beta={p['beta']:.1f}, tau={p['tau']:.2f}): "
                      f"GRACE={grace(A0, r0, **p):.3f}")

for w in (s_beta, s_tau, s_A0, s_r0, s_rmax, c_regimes):
    w.observe(draw_landscape, names="value")
draw_landscape()
display(widgets.HBox([controls, out1]))

## 2. Time-penalty curve $T^*(r)$

$T^*(r)$ is the "real-time ability" factor: full credit at $r=0$, decaying as the
agent uses more time, with an extra exponential penalty once it blows the
deadline ($r>1$). $\tau$ controls how sharp that post-deadline cliff is.

Use this to pick $\tau$ from a concrete policy such as *"at 20% overtime the
agent should keep only half its time credit."*

In [ ]:
# --- Interactive T*(r) curve ------------------------------------------
t_tau = widgets.FloatSlider(value=0.43, min=0.0, max=12.0, step=0.05,
            description="tau", continuous_update=True, readout_format=".2f")
out2 = widgets.Output()

def draw_tstar(*_):
    tau = t_tau.value
    r = np.linspace(0, 3, 300)
    with out2:
        out2.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.6, 3.8))
        ax.plot(r, t_star_vec(r, tau), color="C0", lw=2, label=f"tau={tau:.2f}")
        ax.plot(r, t_star_vec(r, 0.0), color="gray", lw=1, ls=":",
                label="tau=0 (no deadline penalty)")
        ax.axvline(1.0, color="C3", ls="--", lw=1, alpha=0.7)
        ax.text(1.02, 0.9, "deadline", color="C3", fontsize=8)
        ax.set_xlabel("normalized runtime  r"); ax.set_ylabel("T*(r)")
        ax.set_ylim(0, 1.02); ax.set_title("Real-time ability T*(r)")
        ax.legend(fontsize=8); plt.tight_layout(); plt.show()

        print("Deadline penalty factor exp(-tau*(r-1))  (the tau-controlled term")
        print("only; the full T*(r) also carries the 1/(1+r^2) decay):")
        for over in (0.10, 0.20, 0.50):
            full_ratio = t_star(1 + over, tau) / t_star(1.0, tau)
            print(f"  at {over*100:4.0f}% overtime (r={1+over:.2f}): "
                  f"penalty factor {math.exp(-tau*over)*100:5.1f}%   "
                  f"(full T* is {full_ratio*100:5.1f}% of its on-deadline value)")
        # Suggest tau for a 'half credit at X% overtime' policy.
        print("\nTo set tau from a policy 'keep fraction f at X% overtime':  "
              "tau = -ln(f) / X")
        print("  e.g. half credit at 20% overtime -> "
              f"tau = {-math.log(0.5)/0.20:.2f}")

t_tau.observe(draw_tstar, names="value")
draw_tstar()
display(widgets.HBox([widgets.VBox([widgets.HTML('<b>Deadline strictness</b>'),
                                    t_tau], layout=widgets.Layout(width='320px')),
                      out2]))

## 3. Trade-off (indifference) curves

Everything on a single contour line scores the *same* GRACE value: these are the
$(A, r)$ pairs the metric treats as **equally good**. Pick a reference operating
point and the cell draws the indifference curve through it, then answers the
practitioner question *"if accuracy drops by $\Delta A$, how much faster must the
method be to keep the same score?"* -- a direct, interpretable check on whether
your $(\beta,\tau)$ encodes the trade-off you actually want.

In [ ]:
# --- Interactive indifference curve -----------------------------------
i_beta = widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.05,
            description="beta", continuous_update=True, readout_format=".2f")
i_tau  = widgets.FloatSlider(value=0.0, min=0.0, max=8.0, step=0.05,
            description="tau", continuous_update=True, readout_format=".2f")
i_Aref = widgets.FloatSlider(value=0.80, min=0.05, max=1.0, step=0.01,
            description="A_ref", continuous_update=True, readout_format=".2f")
i_rref = widgets.FloatSlider(value=1.00, min=0.05, max=3.0, step=0.05,
            description="r_ref", continuous_update=True, readout_format=".2f")
out3 = widgets.Output()

def find_r_for_grace(target, A, beta, tau, hi=8.0):
    # GRACE decreases monotonically in r, so its maximum for this A is at r=0;
    # if even instant completion can't reach the target, it is infeasible.
    if grace(A, 0.0, beta, tau) < target:
        return None
    lo = 0.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if grace(A, mid, beta, tau) > target:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

def draw_indiff(*_):
    beta, tau = i_beta.value, i_tau.value
    Aref, rref = i_Aref.value, i_rref.value
    g_ref = grace(Aref, rref, beta, tau)
    A = np.linspace(0.01, 1, 60); r = np.linspace(0.01, 3, 80)
    RR, AA = np.meshgrid(r, A)       # rows indexed by A, columns by r
    Z = grace_vec(AA, RR, beta, tau)
    with out3:
        out3.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.6, 4.6))
        cf = ax.contourf(RR, AA, Z, levels=np.linspace(0, 1, 21),
                         cmap="inferno", norm=NORM)
        cs = ax.contour(RR, AA, Z, levels=[g_ref], colors="cyan", linewidths=2.2)
        ax.clabel(cs, fmt=lambda v: f"GRACE={v:.3f}", fontsize=9)
        ax.scatter([rref], [Aref], s=170, marker="*", color="cyan",
                   edgecolor="black", zorder=5)
        fig.colorbar(cf, ax=ax, label="GRACE score")
        ax.set_xlabel("normalized runtime  r"); ax.set_ylabel("accuracy  A")
        ax.set_title(f"Indifference curve through (A={Aref:.2f}, r={rref:.2f})")
        plt.tight_layout(); plt.show()

        print(f"Reference (A={Aref:.2f}, r={rref:.2f}) scores GRACE={g_ref:.3f}.")
        print("To keep that score after an accuracy drop, runtime must improve to:")
        print(f"  {'dA':>5s} {'new A':>7s} {'needed r':>10s} {'speedup':>9s}")
        for dA in (0.05, 0.10, 0.15, 0.20):
            An = Aref - dA
            if An <= 0:
                continue
            rn = find_r_for_grace(g_ref, An, beta, tau)
            if rn is None:
                print(f"  {-dA:>5.2f} {An:>7.2f} {'infeasible':>10s} {'--':>9s}")
            elif rn < 1e-2:
                print(f"  {-dA:>5.2f} {An:>7.2f} {'~0 (instant)':>10s} {'huge':>9s}")
            else:
                print(f"  {-dA:>5.2f} {An:>7.2f} {rn:>10.3f} {rref/rn:>8.2f}x")

for w in (i_beta, i_tau, i_Aref, i_rref):
    w.observe(draw_indiff, names="value")
draw_indiff()
display(widgets.HBox([widgets.VBox([widgets.HTML('<b>Parameters</b>'), i_beta, i_tau,
                                    widgets.HTML('<b>Reference point</b>'), i_Aref, i_rref],
                                   layout=widgets.Layout(width='320px')), out3]))

## 4. Export your chosen parameters

Set the two sliders to the values you settled on above. The cell prints a
ready-to-paste snippet and reports the closest named regime, so you can either
reuse a preset or report your custom $(\beta, \tau)$.

In [ ]:
# --- Export -----------------------------------------------------------
e_beta = widgets.FloatSlider(value=1.0, min=0.1, max=4.0, step=0.05,
            description="beta", readout_format=".2f")
e_tau  = widgets.FloatSlider(value=0.0, min=0.0, max=8.0, step=0.05,
            description="tau", readout_format=".2f")
out4 = widgets.Output()

def draw_export(*_):
    beta, tau = e_beta.value, e_tau.value
    with out4:
        out4.clear_output(wait=True)
        # Closest named regime by (beta, tau) Euclidean distance.
        nearest = min(REGIMES.items(),
                      key=lambda kv: (kv[1]['beta']-beta)**2 + (kv[1]['tau']-tau)**2)
        wb = beta ** 2 / (1.0 + beta ** 2)
        print("Your GRACE configuration")
        print("------------------------")
        print(f"  beta = {beta:.2f}   tau = {tau:.2f}")
        print(f"  interpretation: {'accuracy-leaning' if beta < 0.95 else 'speed-leaning' if beta > 1.05 else 'balanced'}"
              f" (speed-weight w_beta={wb:.2f}),"
              f" {'permissive' if tau < 0.1 else 'strict' if tau > 3 else 'moderate'} deadline")
        print(f"  closest named regime: '{nearest[0]}' "
              f"(beta={nearest[1]['beta']:.1f}, tau={nearest[1]['tau']:.2f})")
        print("\n# Paste into your evaluation code:")
        print("from grace_metric import grace")
        print(f"score = grace(A, r, beta={beta:.2f}, tau={tau:.2f})")

for w in (e_beta, e_tau):
    w.observe(draw_export, names="value")
draw_export()
display(widgets.HBox([widgets.VBox([widgets.HTML('<b>Final choice</b>'), e_beta, e_tau],
                                   layout=widgets.Layout(width='320px')), out4]))